## 리듀서(Reducer) - 상태 업데이트
- 리듀서는 LangGraph에서 상태 업데이트 로직을 정의하는 핵심 메커니즘
- **각 노드가 반환하는 업데이트를 기존 상태에 어떻게 적용할지 결정하는 함수**
- **리듀서의 필요성**
  - 여러 노드가 동시에 또는 순차적으로 실행될 때, 각 노드의 출력을 상태에 통합하는 방법이 필요
  - 단순히 덮어쓰기만 하면 이전 정보가 손실될 수 있으며, 리듀서를 통해 누적, 병합, 최대값 유지 등 다양한 업데이트 전략을 구현할 수 있음
- **노드는 전체 상태가 아닌 업데이트할 부분만 반환하며, 리듀서는 이 부분 업데이트를 기존 상태와 결합**

<br>

#### 기본 동작 원리

```python
# 리듀서 없음 = 덮어쓰기 (Default Reducer)
old_state = {"counter": 5}
new_update = {"counter": 10}
result = {"counter": 10}  # 기존 값이 완전히 대체됨

# 리듀서 있음 = 사용자 정의 동작
from operator import add
old_state = {"items": [1, 2, 3]}
new_update = {"items": [4, 5]}
result = {"items": [1, 2, 3, 4, 5]}  # 리스트가 연결됨
```

<br>

### 기본 리듀서

<br>

#### 덮어쓰기 (Default)


In [22]:
from typing_extensions import TypedDict

In [23]:
class SimpleState(TypedDict):
    counter: int          # 기본 리듀서: 덮어쓰기
    current_user: str     # 기본 리듀서: 덮어쓰기
    status: str          # 기본 리듀서: 덮어쓰기

In [24]:
def update_counter(state: SimpleState) -> SimpleState:
    """
    이 노드는 counter 값만 업데이트
    다른 키들(current_user, status)은 변경되지 않음
    """
    return {"counter": state["counter"] + 1}

In [25]:
def update_multiple(state: SimpleState) -> SimpleState:
    """
    여러 키를 동시에 업데이트할 수 있음
    반환하지 않은 키는 그대로 유지.
    """
    return {
        "counter": 0,           # counter를 0으로 리셋
        "status": "completed"   # status를 "completed"로 변경
    }

<br>

### 내장 리듀서

<br>

#### `operator.add` - 리스트/문자열 추가

In [26]:
from operator import add
from typing import Annotated

In [27]:
class AddState(TypedDict):
    # Annotated[타입, 리듀서함수] 형식으로 리듀서 지정
    messages: Annotated[list[str], add]      # 리스트에 추가
    log_text: Annotated[str, add]            # 문자열 연결

In [28]:
def add_message(state: AddState) -> AddState:
    """
    add 리듀서의 동작:
    - 리스트: 기존_리스트 + 새_리스트
    - 문자열: 기존_문자열 + 새_문자열
    """
    return {
        "messages": ["새 메시지"],              
        "log_text": "새 로그 항목\n"            
    }

In [29]:
initial_state = {
    "messages": ["안녕하세요"],
    "log_text": "시작\n"
}

- `add_message` 노드 실행 후

```python
{
    "messages": ["안녕하세요", "새 메시지"],
    "log_text": "시작\n새 로그 항목\n"
}
```

<br>

#### `add_messages` : 메시지 전용 리듀서
- LangChain 메시지 객체를 위한 특별한 리듀서

In [30]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from typing import Annotated

In [31]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # 메시지 전용 리듀서

In [32]:
def process_user_input(state: ChatState) -> ChatState:
    """
    add_messages 리듀서의 특징:
    1. 새 메시지는 리스트에 추가
    2. 동일한 ID를 가진 메시지는 업데이트
    3. 자동으로 메시지 객체로 역직렬화
    """
    return {
        "messages": [HumanMessage(content="사용자 입력", id="msg_001")]
    }

<br>

- **혹은 `MessageState` 사용**

In [33]:
from langgraph.graph import MessagesState

In [34]:
class MyState(MessagesState):
    """MessagesState를 상속받으면 messages 필드와 add_messages 리듀서가 자동 포함"""
    documents: list[str]  # 추가 필드 정의 가능

<br>

### 사용자 정의 리듀서
- 특정 비즈니스 로직을 구현하는 리듀서

In [35]:
def max_reducer(existing: int, new: int) -> int:
    """
    더 큰 값을 유지하는 리듀서

    Args:
        existing: 현재 상태의 값
        new: 노드가 반환한 새 값

    Returns:
        둘 중 더 큰 값
    """
    return max(existing or 0, new or 0)

In [36]:
class MaxState(TypedDict):
    highest_score: Annotated[int, max_reducer]
    current_score: int  # 리듀서 없음 (덮어쓰기)

<br>

- **`update_score` 실행 후:**
  - `{"highest_score": 90, "current_score": 85}`
  
  $\rightarrow$ `highest_score`는 90이 더 크므로 유지, `current_score`는 85로 덮어쓰기

In [37]:
def update_score(state: MaxState) -> MaxState:
    new_score = 85
    return {
        "highest_score": new_score,  # max_reducer가 적용됨
        "current_score": new_score   # 단순 덮어쓰기
    }

<br>

### 복잡한 사용자 정의 리듀서
- 딕셔너리 병합

In [38]:
def merge_dict_reducer(existing: dict, new: dict) -> dict:
    """
    딕셔너리를 깊게 병합하는 리듀서
    중첩된 딕셔너리도 재귀적으로 병합
    """
    if not existing:
        return new or {}
    if not new:
        return existing

    result = existing.copy()
    for key, value in new.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            # 중첩된 딕셔너리는 재귀적으로 병합
            result[key] = merge_dict_reducer(result[key], value)
        else:
            # 그 외의 경우는 덮어쓰기
            result[key] = value
    return result

In [39]:
class ConfigState(TypedDict):
    settings: Annotated[dict, merge_dict_reducer]

In [40]:
def update_ui_settings(state: ConfigState) -> ConfigState:
    """UI 설정만 업데이트"""
    return {
        "settings": {
            "ui": {"theme": "dark"}  # ui.theme만 변경
        }
    }

In [41]:
def update_performance_settings(state: ConfigState) -> ConfigState:
    """성능 설정만 업데이트"""
    return {
        "settings": {
            "performance": {"cache_size": 1000}  # performance.cache_size만 변경
        }
    }

<br>

- **실행 시나리오**

1. 초기 상태: `{"settings": {"ui": {"theme": "light", "font": "Arial"}, "performance": {"cache_size": 500}}}`
2. `update_ui_settings` 실행 후:
- `{"settings": {"ui": {"theme": "dark", "font": "Arial"}, "performance": {"cache_size": 500}}}` $\rightarrow$ `ui.theme`만 변경되고 나머지는 유지


<br>

### 고급 리듀서 패턴
- **리스트 내 객체를 ID로 식별하여 업데이트하는 패턴**

<br>


In [42]:
def update_by_id_reducer(existing: list[dict], new: list[dict]) -> list[dict]:
    """
    ID를 기준으로 리스트의 항목을 업데이트하는 리듀서

    동작 방식:
    1. 기존 리스트를 ID별로 인덱싱
    2. 새 항목이 같은 ID를 가지면 업데이트
    3. 새 ID면 추가
    """
    if not existing:
        existing = []
    if not new:
        return existing

    # 기존 항목들을 ID로 인덱싱 (딕셔너리로 변환)
    existing_dict = {item.get("id"): item for item in existing}

    # 새 항목들 처리
    for item in new:
        item_id = item.get("id")
        if item_id:
            existing_dict[item_id] = item  # 업데이트 또는 추가

    return list(existing_dict.values())

In [43]:
class EntityState(TypedDict):
    users: Annotated[list[dict], update_by_id_reducer]

In [44]:
def update_user(state: EntityState) -> EntityState:
    """특정 사용자 정보 업데이트"""
    return {
        "users": [
            {"id": "user1", "name": "김철수", "age": 31},  # user1 업데이트
            {"id": "user3", "name": "박영수", "age": 28}   # user3 추가
        ]
    }

- **실행 예시**:
1. 기존 상태:
    
```python
{"users": [
   {"id": "user1", "name": "김철수", "age": 30},
   {"id": "user2", "name": "이영희", "age": 25}
]}
```

2. `update_user` 실행 후:

```python
{"users": [
    {"id": "user1", "name": "김철수", "age": 31},  # age가 31로 업데이트
    {"id": "user2", "name": "이영희", "age": 25},  # 변경 없음
    {"id": "user3", "name": "박영수", "age": 28}   # 새로 추가
]}
```

<br>

### 상태 업데이트

<br>

#### 기본 상태 관리 예시

In [45]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

- 상태 정의

In [46]:
class State(TypedDict):
    message_count: int                           # 기본 리듀서 (덮어쓰기)
    conversation: Annotated[list[str], add]      # add 리듀서 (리스트 연결)

- 노드 함수 정의

In [47]:
def node_1(state: State) -> State:
    """
    첫 번째 노드: 대화 시작
    - message_count를 1로 설정
    - conversation에 인사말 추가
    """
    print(f"Node 1 - 현재 대화: {state.get('conversation', [])}")
    return {
        "message_count": 1,
        "conversation": ["안녕하세요!"]
    }

In [48]:
def node_2(state: State) -> State:
    """
    두 번째 노드: 대화 이어가기
    - message_count는 업데이트하지 않음 (이전 값 유지)
    - conversation에 새 메시지 추가 (add 리듀서로 누적)
    """
    print(f"Node 2 - 현재 대화: {state['conversation']}")
    # message_count를 반환하지 않으므로 node_1에서 설정한 값이 유지됨
    return {
        "conversation": ["어떻게 도와드릴까요?"]
    }

<br>

- 그래프 구성

In [ ]:
graph = StateGraph(State)
graph.add_node("node_1", node_1)
graph.add_node("node_2", node_2)

graph.add_edge(START, "node_1")  # 시작 -> node_1
graph.add_edge("node_1", "node_2")  # node_1 -> node_2
graph.add_edge("node_2", END)  # node_2 -> 종료

compiled_graph = graph.compile()

<br>

- 실행

In [51]:
initial_state = {"message_count": 0, "conversation": []}
result = compiled_graph.invoke(initial_state)

Node 1 - 현재 대화: []
Node 2 - 현재 대화: ['안녕하세요!']


In [52]:
print("\n=== 최종 결과 ===")
print(f"메시지 수: {result['message_count']}")
print(f"대화 내용: {result['conversation']}")


=== 최종 결과 ===
메시지 수: 1
대화 내용: ['안녕하세요!', '어떻게 도와드릴까요?']


<br>

#### 복합 리듀서 활용

In [62]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

In [63]:
def keep_last_n(existing: list, new: list, n: int = 5) -> list:
    """최근 n개 항목만 유지하는 리듀서"""
    combined = (existing or []) + (new or [])
    return combined[-n:]  # 마지막 n개만 반환

- 부분 함수로 리듀서 생성

In [64]:
from functools import partial

In [65]:
keep_last_5 = partial(keep_last_n, n=5)

- 상태 정의

In [72]:
def update_max(left: float, right: float) -> float:
    # 최초 값(None 등) 처리를 위해 max를 안전하게 사용
    if left is None:
        return right
    return max(left, right)

In [73]:
class AdvancedState(TypedDict):
    total_tokens: Annotated[int, add]           # 토큰 수 누적
    max_score: Annotated[float, update_max]            # 최고 점수 유지
    recent_actions: Annotated[list, keep_last_5] # 최근 5개 액션만 유지
    current_status: str    

- 노드 함수 정의

In [74]:
def process_input(state: AdvancedState) -> AdvancedState:
    """사용자 입력 처리"""
    return {
        "total_tokens": 150,  # 150 토큰 추가
        "max_score": 0.85,    # 점수 0.85
        "recent_actions": ["input_received"],
        "current_status": "processing"
    }

In [75]:
def analyze_content(state: AdvancedState) -> AdvancedState:
    """콘텐츠 분석"""
    return {
        "total_tokens": 200,  # 200 토큰 추가
        "max_score": 0.92,    # 더 높은 점수
        "recent_actions": ["analysis_started", "analysis_completed"],
        "current_status": "analyzed"
    }

In [76]:
def generate_response(state: AdvancedState) -> AdvancedState:
    """응답 생성"""
    return {
        "total_tokens": 300,  # 300 토큰 추가
        "max_score": 0.88,    # 낮은 점수 (무시됨)
        "recent_actions": ["response_generated"],
        "current_status": "completed"
    }

- 그래프 구성 및 실행

In [77]:
graph = StateGraph(AdvancedState)
graph.add_node("process", process_input)
graph.add_node("analyze", analyze_content)
graph.add_node("generate", generate_response)

In [78]:
graph.add_edge(START, "process")
graph.add_edge("process", "analyze")
graph.add_edge("analyze", "generate")
graph.add_edge("generate", END)

In [79]:
compiled_graph = graph.compile()

- 실행

In [80]:
result = compiled_graph.invoke({
    "total_tokens": 0,
    "max_score": 0.0,
    "recent_actions": [],
    "current_status": "idle"
})

In [81]:
print("=== 최종 상태 ===")
print(f"총 토큰 수: {result['total_tokens']}")  # 650 (150+200+300)
print(f"최고 점수: {result['max_score']}")      # 0.92 (최대값)
print(f"최근 액션: {result['recent_actions']}")  # 최근 5개
print(f"현재 상태: {result['current_status']}")  # "completed"

=== 최종 상태 ===
총 토큰 수: 650
최고 점수: 0.92
최근 액션: ['input_received', 'analysis_started', 'analysis_completed', 'response_generated']
현재 상태: completed


<br>

<hr>